# Measuring ambiguity in public market data

## Monthly Priors from a Public Daily Series

For year \(y\) and month \(m\), the monthly mean and population standard deviation are

$$
\mu_{y,m}=\frac{1}{n_{y,m}}\sum_{t\in\mathcal I_{y,m}}r_t,
\qquad
\sigma_{y,m}=\left[\frac{1}{n_{y,m}}\sum_{t\in\mathcal I_{y,m}}(r_t-\mu_{y,m})^2\right]^{1/2}.
$$

Here, \(\mathcal I_{y,m}\) is the set of valid daily returns in month \(m\) of year \(y\); \(n_{y,m}\) is its size; \(r_t\) is the daily log return; \(\mu_{y,m}\) is its mean; and \(\sigma_{y,m}\) is its population standard deviation. The twelve fitted normal laws are supplied to the ambiguity calculation with equal default weights. They are teaching approximations, not observations of investor beliefs.

In [1]:
from pathlib import Path
import calendar
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
from scipy.stats import norm
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
from RiskLabAI.features.entropy_features.ambiguity import normal_prior_bin_probabilities, probability_ambiguity
assert RiskLabAI.__version__ == '3.2.0'
DATA_PATH = ensure_dataset(BOOK_ROOT, 'fred_nasdaq')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '04_probability_ambiguity'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '04_probability_ambiguity'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '04_probability_ambiguity'
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    shown = frame.apply(lambda column: column.map(rendered_value))
    for column in frame.columns:
        if 'year' in str(column).lower():
            shown[column] = frame[column].map(lambda value: '' if pd.isna(value) else str(int(value)) if isinstance(value, (int, float, np.integer, np.floating)) and float(value).is_integer() else str(value))
    return shown

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)

def load_public_data():
    assert DATA_PATH.is_file()
    raw = pd.read_csv(DATA_PATH)
    raw['Date'] = pd.to_datetime(raw['observation_date'])
    raw['NASDAQ Composite'] = pd.to_numeric(raw['NASDAQCOM'], errors='coerce')
    data = raw.dropna(subset=['NASDAQ Composite']).sort_values('Date').reset_index(drop=True)
    data['Daily log return'] = np.log(data['NASDAQ Composite'] / data['NASDAQ Composite'].shift(1))
    data['Year'] = data['Date'].dt.year
    data['Month'] = data['Date'].dt.month
    return data

def monthly_moments(data):
    returns = data.dropna(subset=['Daily log return']).copy()
    moments = returns.groupby(['Year', 'Month'])['Daily log return'].agg(Mean='mean', Standard_deviation=lambda values: values.std(ddof=0), Observations='size').reset_index()
    return moments

def normal_edges(width):
    intervals = int(round(0.12 / width))
    return np.linspace(-0.06, 0.06, intervals + 1)

def bin_labels(edges):
    labels = [f'Below {100 * edges[0]:g}%']
    labels.extend((f'{100 * left:g}% to {100 * right:g}%' for left, right in zip(edges[:-1], edges[1:])))
    labels.append(f'Above {100 * edges[-1]:g}%')
    return labels


In [2]:
data = load_public_data()
returns = data.dropna(subset=['Daily log return']).copy()
moments = monthly_moments(data)
provenance = pd.DataFrame({'Item': ['Series', 'Provider', 'Source institution', 'Frequency', 'Local period', 'Download during run'], 'Value': ['NASDAQ Composite (NASDAQCOM)', 'Federal Reserve Economic Data', 'Nasdaq, Inc.', 'Daily close', f"{data['Date'].min().date()} to {data['Date'].max().date()}", 'No']})
save_table(provenance, 'data_provenance', 'Public data and fixed local file identity.', 'tab:chapter5-ambiguity-data-provenance', fit=True)
data_summary = pd.DataFrame({'Quantity': ['Valid index levels', 'Daily log returns', 'Calendar months', 'Complete calendar years', 'Fewest returns in a month', 'Most returns in a month'], 'Value': [len(data), len(returns), len(moments), moments['Year'].nunique(), moments['Observations'].min(), moments['Observations'].max()]})
save_table(data_summary, 'public_data_summary', 'Counts in the public daily NASDAQ Composite sample.', 'tab:chapter5-public-data-summary')
history = data[['Date', 'NASDAQ Composite', 'Daily log return']].copy()
history['Daily log return (%)'] = 100 * history['Daily log return']
history.to_csv(FIGURE_DATA / 'public_price_return_history.csv', index=False)


,Item,Value
0,Series,NASDAQ Composite (NASDAQCOM)
1,Provider,Federal Reserve Economic Data
2,Source institution,"Nasdaq, Inc."
3,Frequency,Daily close
4,Local period,1994-01-03 to 2025-12-31
5,Download during run,No


,Quantity,Value
0,Valid index levels,"8,055"
1,Daily log returns,"8,054"
2,Calendar months,384
3,Complete calendar years,32
4,Fewest returns in a month,15
5,Most returns in a month,23


In [3]:
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.0), sharex=True, constrained_layout=True)
axes[0].plot(history["Date"], history["NASDAQ Composite"], color=PALETTE["cyan"], linewidth=1.35)
axes[0].set_yscale("log")
axes[0].set_ylabel("Index level")
axes[0].set_title("Public NASDAQ Composite Prices and Daily Returns")
axes[1].plot(history["Date"], history["Daily log return (%)"], color=PALETTE["amber"], linewidth=0.62)
axes[1].set_ylabel("Daily log return (%)")
axes[1].set_xlabel("Date")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "public_price_return_history")
display(figure)
plt.close(figure)

<Figure size 1120x700 with 2 Axes>

## Why ambiguity and return risk describe different features

The annual daily return variance is

$$
\sigma_y^2=\frac{1}{n_y}\sum_{t\in\mathcal I_y}(r_t-\bar r_y)^2,
\qquad
\bar r_y=\frac{1}{n_y}\sum_{t\in\mathcal I_y}r_t.
$$

Here, \(\mathcal I_y\) is the set of daily returns in year \(y\); \(n_y\) is the number of those returns; \(\bar r_y\) is their mean; and \(\sigma_y\) is their population standard deviation. This risk summary concerns variation in realized daily outcomes. The ambiguity summary concerns disagreement among twelve fitted monthly probability laws. Neither quantity determines the other.

In [4]:
edges = normal_edges(0.002)
annual_rows = []
for year, year_moments in moments.groupby("Year"):
    year_moments = year_moments.sort_values("Month")
    assert len(year_moments) == 12
    probabilities = normal_prior_bin_probabilities(year_moments["Mean"], year_moments["Standard_deviation"], edges=edges)
    ambiguity = probability_ambiguity(probabilities, bin_width=0.002)
    year_returns = returns.loc[returns["Year"] == year, "Daily log return"].to_numpy()
    annual_rows.append({
        "Year": int(year),
        "Monthly laws": len(year_moments),
        "Daily returns": len(year_returns),
        "Unscaled ambiguity": ambiguity["unscaled"],
        "Scaled ambiguity": ambiguity["scaled"],
        "Daily return standard deviation (%)": 100 * np.std(year_returns, ddof=0),
    })
annual = pd.DataFrame(annual_rows)
save_table(annual, "annual_ambiguity_series", "Annual probability ambiguity and daily return risk from the public series.", "tab:chapter5-annual-ambiguity-series", fit=True)
annual.to_csv(FIGURE_DATA / "annual_ambiguity_series.csv", index=False)

largest = annual.nlargest(5, "Scaled ambiguity").assign(Group="Five largest ambiguity values")
smallest = annual.nsmallest(5, "Scaled ambiguity").assign(Group="Five smallest ambiguity values")
extremes = pd.concat([largest, smallest], ignore_index=True)[["Group", "Year", "Scaled ambiguity", "Daily return standard deviation (%)"]]
save_table(extremes, "descriptive_ambiguity_extremes", "Years with the five largest and five smallest ambiguity values.", "tab:chapter5-descriptive-ambiguity-extremes", fit=True)

correlation = annual["Scaled ambiguity"].corr(annual["Daily return standard deviation (%)"])
relation = pd.DataFrame({
    "Quantity": ["Years", "First year", "Last year", "Correlation between annual summaries", "Largest ambiguity year", "Largest risk year"],
    "Value": [
        len(annual),
        str(int(annual["Year"].min())),
        str(int(annual["Year"].max())),
        correlation,
        str(int(annual.loc[annual["Scaled ambiguity"].idxmax(), "Year"])),
        str(int(annual.loc[annual["Daily return standard deviation (%)"].idxmax(), "Year"])),
    ],
})
save_table(relation, "risk_ambiguity_relation", "Descriptive relation between annual return risk and probability ambiguity.", "tab:chapter5-risk-ambiguity-relation", fit=True)

,Year,Monthly laws,Daily returns,Unscaled ambiguity,Scaled ambiguity,Daily return standard deviation (%)
0,1994,12,251,0.000798,0.399782,0.729229
1,1995,12,252,0.000556,0.278463,0.838085
2,1996,12,254,0.000344,0.17238,0.969899
3,1997,12,253,0.000215,0.107793,1.16458
4,1998,12,252,0.000243,0.12153,1.669583
5,1999,12,252,5.24973e-05,0.026301,1.714988
6,2000,12,252,0.000144,0.072294,3.065
7,2001,12,248,7.8072e-05,0.039114,2.722746
8,2002,12,252,3.48822e-05,0.017476,2.158531
9,2003,12,252,3.11102e-05,0.015586,1.602832


,Group,Year,Scaled ambiguity,Daily return standard deviation (%)
0,Five largest ambiguity values,2014,0.427694,0.890644
1,Five largest ambiguity values,1994,0.399782,0.729229
2,Five largest ambiguity values,2019,0.305016,0.983242
3,Five largest ambiguity values,2016,0.290353,1.002518
4,Five largest ambiguity values,1995,0.278463,0.838085
5,Five smallest ambiguity values,2003,0.015586,1.602832
6,Five smallest ambiguity values,2022,0.016376,2.009704
7,Five smallest ambiguity values,2002,0.017476,2.158531
8,Five smallest ambiguity values,1999,0.026301,1.714988
9,Five smallest ambiguity values,2001,0.039114,2.722746


,Quantity,Value
0,Years,32
1,First year,1994
2,Last year,2025
3,Correlation between annual summaries,-0.549449
4,Largest ambiguity year,2014
5,Largest risk year,2000


,Quantity,Value
0,Years,32
1,First year,1994
2,Last year,2025
3,Correlation between annual summaries,-0.549449
4,Largest ambiguity year,2014
5,Largest risk year,2000


In [5]:
figure, axes = plt.subplots(2, 1, figsize=(11.0, 7.2), sharex=True, constrained_layout=True)
axes[0].plot(annual["Year"], annual["Scaled ambiguity"], color=PALETTE["cyan"], marker="o", markersize=4.0, linewidth=1.6)
axes[0].set_ylabel("Scaled ambiguity")
axes[0].set_title("Probability Ambiguity and Daily Return Risk by Year")
axes[1].plot(annual["Year"], annual["Daily return standard deviation (%)"], color=PALETTE["amber"], marker="o", markersize=4.0, linewidth=1.6)
axes[1].set_ylabel("Daily standard deviation (%)")
axes[1].set_xlabel("Year")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "annual_ambiguity_and_risk")
display(figure)
plt.close(figure)

<Figure size 1100x720 with 2 Axes>

In [6]:
figure, axis = plt.subplots(figsize=(9.4, 5.8), constrained_layout=True)
points = axis.scatter(annual["Daily return standard deviation (%)"], annual["Scaled ambiguity"], c=annual["Year"], cmap="viridis", s=58)
for _, row in annual.loc[annual["Year"].isin([1994, 2000, 2008, 2014, 2020, 2025])].iterrows():
    axis.annotate(str(int(row["Year"])), (row["Daily return standard deviation (%)"], row["Scaled ambiguity"]), xytext=(5, 5), textcoords="offset points")
axis.set_xlabel("Daily return standard deviation (%)")
axis.set_ylabel("Scaled ambiguity")
axis.set_title("Two Descriptive Summaries of Each Calendar Year")
colorbar = figure.colorbar(points, ax=axis)
colorbar.set_label("Year")
finish_axis(axis)
save_figure(figure, "risk_ambiguity_scatter")
display(figure)
plt.close(figure)

<Figure size 940x580 with 2 Axes>

## Testing sensitivity to grid and weights

For a chosen interior width \(h\), the annual scaled value is

$$
A_y^{\mathrm{scaled}}(h)=\frac{\sum_k\bar p_{y,k}(h)v_{y,k}(h)}{h(1-h)}.
$$

Here, \(A_y^{\mathrm{scaled}}(h)\) is the scaled ambiguity value for year \(y\); \(\bar p_{y,k}(h)\) and \(v_{y,k}(h)\) are computed on the grid defined by width \(h\); and \(k\) includes the two tails. Values at several widths reveal remaining grid sensitivity.

Observation count weights are

$$
q_{y,m}^{\mathrm{count}}=\frac{n_{y,m}}{\sum_{\ell=1}^{12}n_{y,\ell}}.
$$

Here, \(n_{y,m}\) is the number of daily returns in month \(m\) of year \(y\). Equal weights treat every month alike, while count weights give slightly more weight to months with more valid daily observations. The comparison is a sensitivity check, not a search for a preferred result.

In [7]:
grid_rows = []
weight_rows = []
for year, year_moments in moments.groupby("Year"):
    year_moments = year_moments.sort_values("Month")
    for width in [0.001, 0.002, 0.004]:
        width_edges = normal_edges(width)
        probabilities = normal_prior_bin_probabilities(year_moments["Mean"], year_moments["Standard_deviation"], edges=width_edges)
        ambiguity = probability_ambiguity(probabilities, bin_width=width)
        grid_rows.append({
            "Year": int(year),
            "Interior bin width": width,
            "Interior bins": len(width_edges) - 1,
            "Unscaled ambiguity": ambiguity["unscaled"],
            "Scaled ambiguity": ambiguity["scaled"],
        })
    default_probabilities = normal_prior_bin_probabilities(year_moments["Mean"], year_moments["Standard_deviation"], edges=edges)
    equal = probability_ambiguity(default_probabilities, bin_width=0.002)
    count_weights = year_moments["Observations"].to_numpy(dtype=float)
    count_weights = count_weights / count_weights.sum()
    count = probability_ambiguity(default_probabilities, prior_weights=count_weights, bin_width=0.002)
    weight_rows.append({
        "Year": int(year),
        "Equal month weights": equal["scaled"],
        "Observation count weights": count["scaled"],
        "Absolute difference": abs(equal["scaled"] - count["scaled"]),
    })

grid_sensitivity = pd.DataFrame(grid_rows)
weight_sensitivity = pd.DataFrame(weight_rows)
grid_sensitivity.to_csv(FIGURE_DATA / "grid_sensitivity.csv", index=False)
weight_sensitivity.to_csv(FIGURE_DATA / "weight_sensitivity.csv", index=False)
save_table(grid_sensitivity, "grid_sensitivity_all_years", "Annual ambiguity values under three interior bin widths.", "tab:chapter5-grid-sensitivity-all-years", fit=True)
save_table(weight_sensitivity, "weight_sensitivity_all_years", "Annual ambiguity values under equal and observation count weights.", "tab:chapter5-weight-sensitivity-all-years", fit=True)

grid_summary = grid_sensitivity.groupby(["Interior bin width", "Interior bins"]).agg(
    Mean_scaled_ambiguity=("Scaled ambiguity", "mean"),
    Minimum_scaled_ambiguity=("Scaled ambiguity", "min"),
    Maximum_scaled_ambiguity=("Scaled ambiguity", "max"),
).reset_index()
grid_summary.columns = ["Interior bin width", "Interior bins", "Mean scaled ambiguity", "Minimum scaled ambiguity", "Maximum scaled ambiguity"]
save_table(grid_summary, "grid_sensitivity_summary", "Summary across years for three interior bin widths.", "tab:chapter5-grid-sensitivity-summary", fit=True)

weight_summary = pd.DataFrame({
    "Quantity": ["Mean equal weight value", "Mean observation count weight value", "Mean absolute difference", "Largest absolute difference", "Year of largest difference"],
    "Value": [
        weight_sensitivity["Equal month weights"].mean(),
        weight_sensitivity["Observation count weights"].mean(),
        weight_sensitivity["Absolute difference"].mean(),
        weight_sensitivity["Absolute difference"].max(),
        str(int(weight_sensitivity.loc[weight_sensitivity["Absolute difference"].idxmax(), "Year"])),
    ],
})
save_table(weight_summary, "weight_sensitivity_summary", "Summary of the prior weight sensitivity comparison.", "tab:chapter5-weight-sensitivity-summary", fit=True)

,Year,Interior bin width,Interior bins,Unscaled ambiguity,Scaled ambiguity
0,1994,0.001,120,0.000204,0.20455
1,1994,0.002,60,0.000798,0.399782
2,1994,0.004,30,0.002921,0.733114
3,1995,0.001,120,0.000141,0.141262
4,1995,0.002,60,0.000556,0.278463
...,...,...,...,...,...
91,2024,0.002,60,0.000134,0.067031
92,2024,0.004,30,0.000521,0.13066
93,2025,0.001,120,0.000125,0.124875
94,2025,0.002,60,0.00049,0.245476


,Year,Equal month weights,Observation count weights,Absolute difference
0,1994,0.399782,0.402473,0.002691
1,1995,0.278463,0.278238,0.000224
2,1996,0.17238,0.171575,0.000804
3,1997,0.107793,0.109177,0.001384
4,1998,0.12153,0.118076,0.003454
5,1999,0.026301,0.026312,1.04677e-05
6,2000,0.072294,0.06951,0.002785
7,2001,0.039114,0.038717,0.000397
8,2002,0.017476,0.017533,5.68903e-05
9,2003,0.015586,0.015672,8.54302e-05


,Interior bin width,Interior bins,Mean scaled ambiguity,Minimum scaled ambiguity,Maximum scaled ambiguity
0,0.001,120,0.083679,0.007809,0.218199
1,0.002,60,0.155022,0.015586,0.427694
2,0.004,30,0.290802,0.03086,0.791033


,Quantity,Value
0,Mean equal weight value,0.155022
1,Mean observation count weight value,0.155
2,Mean absolute difference,0.002316
3,Largest absolute difference,0.012405
4,Year of largest difference,2014


,Quantity,Value
0,Mean equal weight value,0.155022
1,Mean observation count weight value,0.155
2,Mean absolute difference,0.002316
3,Largest absolute difference,0.012405
4,Year of largest difference,2014


In [8]:
figure, axis = plt.subplots(figsize=(10.8, 5.7), constrained_layout=True)
for width, color, marker in [(0.001, PALETTE["cyan"], "o"), (0.002, PALETTE["amber"], "s"), (0.004, PALETTE["green"], "^")]:
    part = grid_sensitivity[grid_sensitivity["Interior bin width"] == width]
    axis.plot(part["Year"], part["Scaled ambiguity"], color=color, marker=marker, markersize=3.8, linewidth=1.5, label=f"Width {width:g}")
axis.set_xlabel("Year")
axis.set_ylabel("Scaled ambiguity")
axis.set_title("The Scaling Does Not Remove All Grid Sensitivity")
axis.legend()
finish_axis(axis)
save_figure(figure, "grid_sensitivity")
display(figure)
plt.close(figure)

<Figure size 1080x570 with 1 Axes>

In [9]:
figure, axes = plt.subplots(1, 2, figsize=(11.0, 4.8), constrained_layout=True)
axes[0].plot(weight_sensitivity["Year"], weight_sensitivity["Equal month weights"], color=PALETTE["cyan"], linewidth=1.7, label="Equal month weights")
axes[0].plot(weight_sensitivity["Year"], weight_sensitivity["Observation count weights"], color=PALETTE["amber"], linestyle="--", linewidth=1.7, label="Observation count weights")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Scaled ambiguity")
axes[0].set_title("Two Prior Weight Choices")
axes[0].legend(fontsize=8.5)
axes[1].bar(weight_sensitivity["Year"], weight_sensitivity["Absolute difference"], color=PALETTE["rose"])
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Absolute difference")
axes[1].set_title("Difference Between Weight Choices")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "weight_sensitivity")
display(figure)
plt.close(figure)

<Figure size 1100x480 with 2 Axes>

In [10]:
worked_direct = monthly_moments(data.loc[data["Year"] == 1994].copy())
stored_worked = moments.loc[moments["Year"] == 1994].reset_index(drop=True)
assert np.allclose(worked_direct[["Mean", "Standard_deviation"]], stored_worked[["Mean", "Standard_deviation"]])
assert set(annual["Year"]) == set(range(1994, 2026))
assert annual["Monthly laws"].eq(12).all()
assert annual["Daily returns"].sum() == len(returns)
assert np.all(grid_sensitivity["Scaled ambiguity"] >= 0)
assert np.all(weight_sensitivity[["Equal month weights", "Observation count weights"]] >= 0)

checks = pd.DataFrame({
    "Check": [
        "Public file identity",
        "All years contain twelve monthly laws",
        "Annual rows use only returns from their calendar year",
        "Every daily return enters exactly one annual row",
        "All grid sensitivity values are nonnegative",
        "Both prior weight calculations are nonnegative",
        "The worked year is the first complete year by date",
    ],
    "Result": ["Pass"] * 7,
})
save_table(checks, "public_example_checks", "Checks for the public annual ambiguity example.", "tab:chapter5-public-example-checks", fit=True)

limits = pd.DataFrame({
    "Question": [
        "What is observed?",
        "What is fitted?",
        "What does ambiguity summarize?",
        "What does return risk summarize?",
        "What is not claimed?",
    ],
    "Answer": [
        "Public daily NASDAQ Composite closing levels",
        "One normal daily return law for each calendar month",
        "Dispersion of bin probabilities across twelve supplied monthly laws",
        "Dispersion of realized daily returns within a calendar year",
        "Investor beliefs, causal effects, pricing results, or investment performance",
    ],
})
save_table(limits, "interpretation_limits", "Interpretation boundary for the public daily illustration.", "tab:chapter5-interpretation-limits", fit=True)

,Check,Result
0,Public file identity,Pass
1,All years contain twelve monthly laws,Pass
2,Annual rows use only returns from their calend...,Pass
3,Every daily return enters exactly one annual row,Pass
4,All grid sensitivity values are nonnegative,Pass
5,Both prior weight calculations are nonnegative,Pass
6,The worked year is the first complete year by ...,Pass


,Question,Answer
0,What is observed?,Public daily NASDAQ Composite closing levels
1,What is fitted?,One normal daily return law for each calendar ...
2,What does ambiguity summarize?,Dispersion of bin probabilities across twelve ...
3,What does return risk summarize?,Dispersion of realized daily returns within a ...
4,What is not claimed?,"Investor beliefs, causal effects, pricing resu..."


,Question,Answer
0,What is observed?,Public daily NASDAQ Composite closing levels
1,What is fitted?,One normal daily return law for each calendar ...
2,What does ambiguity summarize?,Dispersion of bin probabilities across twelve ...
3,What does return risk summarize?,Dispersion of realized daily returns within a ...
4,What is not claimed?,"Investor beliefs, causal effects, pricing resu..."
